# ML Learning Coach (English): Part 1, Baseline
Saves how the **untouched base model** (`Qwen/Qwen2.5-1.5B-Instruct`) answers our test prompts, BEFORE fine-tuning.
Runtime > Change runtime type > **T4 GPU**. Run the cells from top to bottom.

### 1) GPU and libraries

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate datasets

### 2) Connect Drive

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/ml-coach-en"
os.makedirs(DRIVE_DIR, exist_ok=True)
print(os.listdir(DRIVE_DIR))

### 3) Get the files
Uses Drive if the files are there, otherwise asks you to upload `train.jsonl` and `test_prompts.json`.

In [ ]:
import shutil
from google.colab import files

def fetch(name):
    if os.path.exists(name):
        return
    p = f"{DRIVE_DIR}/{name}"
    if os.path.exists(p):
        shutil.copy(p, name)
        return
    print(f"{name} not found, please upload it:")
    up = files.upload()
    k = list(up)[0]
    if k != name:
        shutil.move(k, name)
    shutil.copy(name, p)

for name in ["train.jsonl", "test_prompts.json"]:
    fetch(name)
print("Files ready")

### 4) Read the data
The system prompt is taken from `train.jsonl`, so training and testing use exactly the same one.

In [ ]:
import json

with open("train.jsonl", encoding="utf-8") as f:
    rows = [json.loads(l) for l in f if l.strip()]
SYSTEM = rows[0]["messages"][0]["content"]
with open("test_prompts.json", encoding="utf-8") as f:
    tests = json.load(f)
print("Training examples:", len(rows), "| test prompts:", len(tests))

### 5) Load the base model in 4-bit
Training will use the same 4-bit (NF4) loading, so the baseline is measured the same way.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE = "Qwen/Qwen2.5-1.5B-Instruct"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map="auto")
model.eval()
print("GPU memory (GB):", round(torch.cuda.memory_allocated() / 1024**3, 2))

### 6) Answer the test prompts
`do_sample=False` makes the output repeatable, so the before/after comparison is fair.

In [ ]:
def generate(messages, max_new_tokens=300):
    msgs = [{"role": "system", "content": SYSTEM}] + messages
    inputs = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, temperature=None, top_p=None, top_k=None)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


results = []
for t in tests:
    answer = generate(t["messages"])
    results.append({"id": t["id"], "type": t["type"],
                    "last_user": t["messages"][-1]["content"], "answer": answer})
    print("=" * 70)
    print(f"[{t['id']}]  ({t['type']})")
    print("STUDENT:", t["messages"][-1]["content"])
    print("BASE MODEL:\n" + answer)

### 7) Save the results
Keep `baseline_outputs.json`: it is the "before" evidence for the README.

In [ ]:
with open("baseline_outputs.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
shutil.copy("baseline_outputs.json", f"{DRIVE_DIR}/baseline_outputs.json")
files.download("baseline_outputs.json")